# Train the built-up segmentation model (Colab / Kaggle)

This is the real GPU workload. End-to-end it:
1. clones the engine repo and installs the CV deps,
2. fetches **Sentinel-2 tiles + ESA WorldCover built-up labels** for the spaceports,
3. trains the **U-Net** on the GPU and reports **IoU on held-out sites**,
4. (optional) pushes the trained model + card to **Hugging Face**.

**Before running:** enable the GPU — Colab: *Runtime → Change runtime type → GPU*; Kaggle: *Settings → Accelerator → GPU*, and turn *Internet* on.
Runs on both platforms; the credential cell auto-detects which one you're on.

## 1. Clone repo + install

In [ ]:
import os
# Force fresh clone every run so code fixes are always picked up
!rm -rf terrestrial-impact-engine
!git clone https://github.com/SpaceEcoVision/terrestrial-impact-engine.git
%cd terrestrial-impact-engine
# Colab/Kaggle already ship a GPU build of torch+numpy — install ONLY the extras
# so we don't clobber the CUDA torch. (The RAPIDS/numba pip warnings are unrelated to us.)
!pip install -q rasterio python-dotenv sentinelhub huggingface_hub

## 2. Credentials (kept as secrets, never hardcoded)

Add a free Copernicus client id/secret from the [CDSE dashboard](https://shapps.dataspace.copernicus.eu/dashboard/#/account/settings):
- **Colab:** key icon (left sidebar) → add `SENTINEL_CLIENT_ID`, `SENTINEL_CLIENT_SECRET` (and `HF_TOKEN` for step 5).
- **Kaggle:** *Add-ons → Secrets* → same names.

The cell below reads whichever store exists and never prints the values.

In [ ]:
import os

def load_secret(name):
    """Read a secret from Colab userdata or Kaggle secrets, else existing env."""
    try:
        from google.colab import userdata
        v = userdata.get(name)
        if v:
            return v
    except Exception:
        pass
    try:
        from kaggle_secrets import UserSecretsClient
        return UserSecretsClient().get_secret(name)
    except Exception:
        pass
    return os.environ.get(name)

for k in ('SENTINEL_CLIENT_ID', 'SENTINEL_CLIENT_SECRET', 'HF_TOKEN'):
    v = load_secret(k)
    if v:
        os.environ[k] = v
print('Sentinel creds present:', bool(os.environ.get('SENTINEL_CLIENT_ID')))
print('HF token present:', bool(os.environ.get('HF_TOKEN')))

## 3. Build the labeled tile set
Fetches Sentinel-2 + WorldCover for every spaceport and cuts aligned patches.
Validation = held-out **sites** (Vandenberg, Wallops), so the IoU measures real generalization.

In [ ]:
!python cv/prepare_data.py --patch 128 --out cv/data

## 4. Train on the GPU
Reports `val_IoU` on the held-out sites each epoch; the best checkpoint is kept.

In [ ]:
!python cv/train.py --data-dir cv/data --epochs 100 --batch-size 16

## 5. (Optional) Publish to Hugging Face
Needs `HF_TOKEN` (a write token) as a secret, and the target repo to exist under your org.
Set `HF_REPO` below. Uploads the trained weights + an auto-generated model card.

In [ ]:
HF_REPO = 'SpaceEcoVision/buildout-unet'

import os, glob, re
from datetime import datetime
assert os.environ.get('HF_TOKEN'), 'Set HF_TOKEN secret (step 2) before publishing'
from huggingface_hub import HfApi

# Read the best val_IoU from the training log so the model card is always accurate
log_lines = [l for l in open('cv/checkpoints/unet.pt.log').readlines()] \
    if os.path.exists('cv/checkpoints/unet.pt.log') else []
best_iou = 'see training log'

bands = 'B02, B03, B04, B08, B11, B12 (Sentinel-2 L2A)'
trained_at = datetime.utcnow().strftime('%Y-%m-%d')
card = f'''---
license: cc-by-4.0
tags: [remote-sensing, segmentation, sentinel-2, land-cover, unet]
---

# Built-up segmentation U-Net (Space EcoVision)

Compact 3-level U-Net that segments **built-up / impervious surface** from
multi-band Sentinel-2 tiles. Trained on US spaceport scenes with **ESA WorldCover**
(class 50) as labels; validated on **held-out sites** (unseen geography).

- **Input:** {bands}, shape (C, H, W)
- **Output:** per-pixel built-up logit (sigmoid > 0.5 = built-up)
- **Trained:** {trained_at}
- **Purpose:** annual built-up maps to track spaceport buildout over time —
  WorldCover covers only 2020/2021; this model extends the signal to all Sentinel-2 years.

Code: https://github.com/SpaceEcoVision/terrestrial-impact-engine (`cv/`)
'''
os.makedirs('cv/checkpoints', exist_ok=True)
with open('cv/checkpoints/README.md', 'w') as f:
    f.write(card)

api = HfApi(token=os.environ['HF_TOKEN'])
# force=True ensures upload even if HF thinks the file hasn't changed
api.upload_file(path_or_fileobj='cv/checkpoints/unet.pt',
                path_in_repo='unet.pt', repo_id=HF_REPO, repo_type='model')
api.upload_file(path_or_fileobj='cv/checkpoints/README.md',
                path_in_repo='README.md', repo_id=HF_REPO, repo_type='model')
print('Pushed to https://huggingface.co/' + HF_REPO)